# Notebook 4 : Inférence et évaluation

Ce notebook évalue le modèle fine-tuné sur le test set.

Le modèle entraîné est utilisé pour générer des prédictions pour les trois tâches.

Ces prédictions sont ensuite comparées aux annotations de référence issues du dataset agrégé.

**Logique d'évaluation**

- Task A : accuracy, precision, rappel... Seulement sur les tweets qui ont un label de réf stable - les cas ambigus sont exclus de l'accuracy principale 
- Task B : Rouge-L sur les toutes toutes les annotations/seulement les annotation sur la catégories majoritaire. 
- Task C : eval task A -> Eval task B

**Sorties**

Ce notebook produit les résultats d’évaluation, des exemples de prédictions, ainsi qu’un fichier de résumé des métriques, par exemple :

metrics_summary.json

Ces résultats sont ensuite utilisés pour interpréter les performances du modèle

### Chargements 

In [4]:
"!pip install unsloth\n",
"!pip install \"datasets>=3.4.1,<4.4.0,!=4.0.*,!=4.1.0\" \"trl>=0.18.2,<=0.24.0,!=0.19.0\" peft accelerate bitsandbytes scikit-learn pandas tqdm"

'!pip install "datasets>=3.4.1,<4.4.0,!=4.0.*,!=4.1.0" "trl>=0.18.2,<=0.24.0,!=0.19.0" peft accelerate bitsandbytes scikit-learn pandas tqdm'

In [1]:
import unsloth
from unsloth import FastLanguageModel

import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0611 14:00:54.122000 30348 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Directions

In [2]:
PROCESSED_DIR = Path("data/processed")
SFT_DIR = Path("data/sft")

MODEL_DIR = Path("models/enthymemes_unsloth_lora")
LORA_ADAPTER_DIR = MODEL_DIR / "lora_adapter"

EVAL_DIR = Path("evaluation_results")
EVAL_DIR.mkdir(parents=True, exist_ok=True)

TEST_AGGREGATED_PATH = PROCESSED_DIR / "test_aggregated.jsonl"
TEST_SFT_PATH = SFT_DIR / "test_sft.jsonl"

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("LoRA adapter:", LORA_ADAPTER_DIR)

Device: cuda
LoRA adapter: models\enthymemes_unsloth_lora\lora_adapter


### Chargement de test dataset

In [3]:
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    return records

In [4]:
test_records = read_jsonl(TEST_AGGREGATED_PATH)

print("Test aggregated records:", len(test_records))
test_records[0]

Test aggregated records: 134


{'id': '1145',
 'tweet_text': "It isn't funny. Democrats have been forcefully controlling Americans bodies for 2.5 years now with vaccines, passports, masks, and more.",
 'n_annotators': 5,
 'annotations_normalized': [{'annotator_id': 0,
   'label': 'premise',
   'implicit_text': 'If {Democrats have been forcefully controlling Americans bodies for 2.5 years now with vaccines, passports, masks, and more} then "It" isn\'t funny .'},
  {'annotator_id': 1,
   'label': 'premise',
   'implicit_text': 'Forceful control of Americans’ bodies through vaccines, passports, masks, and similar measures for 2.5 years violates human is very wrong and should be taken very seriously.'},
  {'annotator_id': 2,
   'label': 'premise',
   'implicit_text': 'forcefully controlling Americans bodies for 2.5 years now with vaccines, passports, masks, and more viola;tes human rights'},
  {'annotator_id': 3,
   'label': 'premise',
   'implicit_text': "Forcefully controlling Americans' bodies for 2.5 years through v

### Chargement de modèle

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(LORA_ADAPTER_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

FastLanguageModel.for_inference(model)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Model loaded.")

==((====))==  Unsloth 2026.6.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5070. Num GPUs = 1. Max memory: 11.94 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model loaded.


### Prompts pour inference

In [6]:
SYSTEM_PROMPT = """You are an expert annotator of enthymemes and argumentative structures.

Your task is to analyze short social media texts.
You must identify whether the text contains missing implicit argumentation.
If an implicit argument exists, classify it as either:
- implicit premise: a missing supporting reason or assumption
- implicit claim: a missing conclusion inferred from explicit premises

Return only valid JSON.
Do not add explanations outside JSON.
"""

#### Task A prompt

In [7]:
#TASK A: Analyze the enthymeme and classify the missing implicit argumentation.

def build_task_a_prompt(tweet: str) -> List[Dict[str, str]]:
    user_prompt = f"""Task A: Analyze the enthymeme and classify the missing implicit argumentation.

You must decide whether the tweet contains missing implicit argumentation.
If yes, classify the missing argument as:
- premise
- claim

If there is no missing implicit argumentation, use:
- none

If the case is strongly uncertain, mark it as ambiguous.

Tweet:
{tweet}

Return JSON with the following fields:
- argument_analysis
- implicit_status: yes / no / ambiguous
- implicit_type: none / premise / claim / ambiguous
- vote_distribution
- confidence
- uncertainty_level
"""

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

#### Task B prompt

In [8]:
#TASK B: Generate the missing implicit argument.

def build_task_b_prompt(
    tweet: str,
    implicit_type: str,
    uncertainty_level: str = "unknown",
    vote_distribution: Optional[Dict[str, float]] = None,
) -> List[Dict[str, str]]:

    if vote_distribution is None:
        vote_distribution = {
            "none": 0.0,
            "premise": 0.0,
            "claim": 0.0,
        }

    user_prompt = f"""Task B: Generate the missing implicit argument.

The implicit argument type is given.
Generate only the missing proposition that completes the enthymeme.

Tweet:
{tweet}

Implicit argument type:
{implicit_type}

Model-estimated vote distribution:
none: {vote_distribution.get("none", 0.0)}
premise: {vote_distribution.get("premise", 0.0)}
claim: {vote_distribution.get("claim", 0.0)}

Model-estimated uncertainty level:
{uncertainty_level}

Return JSON with the following fields:
- implicit_type
- implicit_argument
"""

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

#### Fonction de génération

In [9]:
def generate_from_messages(
    messages: List[Dict[str, str]],
    max_new_tokens: int = 512,
    temperature: float = 0.1,
    do_sample: bool = False,
) -> str:
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return generated_text.strip()

In [10]:
#json parser 

def try_parse_json(text: str) -> Optional[Dict[str, Any]]:
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        candidate = text[start:end + 1]

        try:
            return json.loads(candidate)
        except Exception:
            return None

    return None

In [11]:
#normalisation des labels

def normalize_predicted_label(label: Any) -> str:
    if label is None:
        return "none"

    label = str(label).strip().lower()

    mapping = {
        "none": "none",
        "no": "none",
        "no implicit": "none",

        "premise": "premise",
        "implicit premise": "premise",
        "implicit_premise": "premise",

        "claim": "claim",
        "conclusion": "claim",
        "implicit claim": "claim",
        "implicit_claim": "claim",
        "implicit conclusion": "claim",

        "ambiguous": "ambiguous",
        "uncertain": "ambiguous",
        "mixed": "ambiguous",
    }

    return mapping.get(label, "ambiguous")

### Check on 1 tweet

In [12]:
example = test_records[0]
tweet = example["tweet_text"]

print(tweet)
print("\nGold label:", example["majority_label"])
print("Gold vote distribution:", example["vote_distribution"])

It isn't funny. Democrats have been forcefully controlling Americans bodies for 2.5 years now with vaccines, passports, masks, and more.

Gold label: premise
Gold vote distribution: {'none': 0.0, 'premise': 1.0, 'claim': 0.0}


In [13]:
messages = build_task_a_prompt(tweet)
raw_output = generate_from_messages(messages)

print(raw_output)

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\jeann\Documents\Projets_Python\Deep_Learning\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
c:\Users\jeann\Documents\Projets_Python\Deep_Learning\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
 

{
  "argument_analysis": "The annotators mostly judged that the tweet does not require an additional hidden premise or hidden claim to complete its argumentative structure.",
  "implicit_status": "no",
  "implicit_type": "none",
  "vote_distribution": {
    "none": 1.0,
    "premise": 0.0,
    "claim": 0.0
  },
  "confidence": 1.0,
  "uncertainty_level": "low"
}


In [14]:
parsed = try_parse_json(raw_output)
parsed

{'argument_analysis': 'The annotators mostly judged that the tweet does not require an additional hidden premise or hidden claim to complete its argumentative structure.',
 'implicit_status': 'no',
 'implicit_type': 'none',
 'vote_distribution': {'none': 1.0, 'premise': 0.0, 'claim': 0.0},
 'confidence': 1.0,
 'uncertainty_level': 'low'}

### Evaluations

#### Task A 

In [15]:
#TASK A : Evaluation

task_a_predictions = []

for record in tqdm(test_records, desc="Task A inference"):
    tweet = record["tweet_text"]

    messages = build_task_a_prompt(tweet)

    raw_output = generate_from_messages(
        messages,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False,
    )

    parsed = try_parse_json(raw_output)

    if parsed is None:
        predicted_type = "parse_error"
        predicted_status = "parse_error"
        predicted_uncertainty = "unknown"
        predicted_confidence = None
        predicted_vote_distribution = None
        argument_analysis = ""
    else:
        predicted_type = normalize_predicted_label(parsed.get("implicit_type", "none"))
        predicted_status = parsed.get("implicit_status", "")
        predicted_uncertainty = parsed.get("uncertainty_level", "unknown")
        predicted_confidence = parsed.get("confidence", None)
        predicted_vote_distribution = parsed.get("vote_distribution", None)
        argument_analysis = parsed.get("argument_analysis", "")

    task_a_predictions.append(
        {
            "id": record["id"],
            "tweet_text": tweet,

            "gold_majority_label": record["majority_label"],
            "gold_implicit_status": record["implicit_status"],
            "gold_uncertainty_level": record["uncertainty_level"],
            "gold_confidence": record["confidence"],
            "gold_vote_distribution": record["vote_distribution"],

            "usable_for_classification": record.get("usable_for_classification", False),

            "predicted_implicit_type": predicted_type,
            "predicted_implicit_status": predicted_status,
            "predicted_uncertainty_level": predicted_uncertainty,
            "predicted_confidence": predicted_confidence,
            "predicted_vote_distribution": predicted_vote_distribution,
            "predicted_argument_analysis": argument_analysis,

            "raw_output": raw_output,
            "parse_success": parsed is not None,
        }
    )

Task A inference:   0%|          | 0/134 [00:00<?, ?it/s]

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

In [16]:
task_a_df = pd.DataFrame(task_a_predictions)

display(task_a_df.head())

print("Parse success:")
display(task_a_df["parse_success"].value_counts())

,id,tweet_text,gold_majority_label,gold_implicit_status,gold_uncertainty_level,gold_confidence,gold_vote_distribution,usable_for_classification,predicted_implicit_type,predicted_implicit_status,predicted_uncertainty_level,predicted_confidence,predicted_vote_distribution,predicted_argument_analysis,raw_output,parse_success
0,1145,It isn't funny. Democrats have been forcefully...,premise,yes,low,1.0,"{'none': 0.0, 'premise': 1.0, 'claim': 0.0}",True,none,no,low,1.0,"{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",The annotators mostly judged that the tweet do...,"{\n ""argument_analysis"": ""The annotators most...",True
1,3298,I sure hope everyone is paying attention.There...,none,no,low,0.8,"{'none': 0.8, 'premise': 0.2, 'claim': 0.0}",True,none,no,low,0.8,"{'none': 0.8, 'premise': 0.2, 'claim': 0.0}",The annotators mostly judged that the tweet do...,"{\n ""argument_analysis"": ""The annotators most...",True
2,3214,Refugees resettled in the UK under the Vulnera...,none,no,low,1.0,"{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",True,none,no,low,1.0,"{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",The annotators mostly judged that the tweet do...,"{\n ""argument_analysis"": ""The annotators most...",True
3,3013,It's revealing that lying about his asylum cla...,claim,yes,low,1.0,"{'none': 0.0, 'premise': 0.0, 'claim': 1.0}",True,none,no,low,0.8,"{'none': 0.8, 'premise': 0.2, 'claim': 0.0}",The annotators mostly judged that the tweet do...,"{\n ""argument_analysis"": ""The annotators most...",True
4,2129,BXP and Tory's should not endorse mass migrati...,none,no,low,1.0,"{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",True,none,no,low,1.0,"{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",The annotators mostly judged that the tweet do...,"{\n ""argument_analysis"": ""The annotators most...",True


Parse success:


parse_success
True    134
Name: count, dtype: int64

In [17]:
#Save task A predictions

task_a_predictions_path = EVAL_DIR / "task_a_predictions.csv"
task_a_df.to_csv(task_a_predictions_path, index=False)

print("Saved:", task_a_predictions_path)

Saved: evaluation_results\task_a_predictions.csv


In [ ]:
#hard classification evaluation : only cases without ambiguity 

hard_eval_df = task_a_df[
    task_a_df["usable_for_classification"] == True
].copy()

hard_eval_df = hard_eval_df[
    hard_eval_df["predicted_implicit_type"].isin(["none", "premise", "claim"])
].copy()

print("Hard eval examples:", len(hard_eval_df))

Hard eval examples: 129


In [19]:
y_true = hard_eval_df["gold_majority_label"].tolist()
y_pred = hard_eval_df["predicted_implicit_type"].tolist()

labels = ["none", "premise", "claim"]

print("Accuracy:", accuracy_score(y_true, y_pred))

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0,
    )
)

Accuracy: 0.6434108527131783

Classification report:
              precision    recall  f1-score   support

        none       0.70      0.85      0.77        87
     premise       0.56      0.24      0.34        37
       claim       0.00      0.00      0.00         5

    accuracy                           0.64       129
   macro avg       0.42      0.36      0.37       129
weighted avg       0.64      0.64      0.62       129



In [20]:
cm = confusion_matrix(y_true, y_pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"gold_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels],
)

cm_df

,pred_none,pred_premise,pred_claim
gold_none,74,7,6
gold_premise,26,9,2
gold_claim,5,0,0


In [21]:
#binary implicit / none eval

def to_binary_label(label: str) -> str:
    if label == "none":
        return "none"
    if label in ["premise", "claim"]:
        return "implicit"
    return "ambiguous"

In [22]:
binary_eval_df = hard_eval_df.copy()

binary_eval_df["gold_binary"] = binary_eval_df["gold_majority_label"].apply(to_binary_label)
binary_eval_df["pred_binary"] = binary_eval_df["predicted_implicit_type"].apply(to_binary_label)

binary_eval_df = binary_eval_df[
    binary_eval_df["pred_binary"].isin(["none", "implicit"])
].copy()

y_true_bin = binary_eval_df["gold_binary"].tolist()
y_pred_bin = binary_eval_df["pred_binary"].tolist()

print("Binary accuracy:", accuracy_score(y_true_bin, y_pred_bin))

print("\nBinary classification report:")
print(
    classification_report(
        y_true_bin,
        y_pred_bin,
        labels=["none", "implicit"],
        zero_division=0,
    )
)

Binary accuracy: 0.6589147286821705

Binary classification report:
              precision    recall  f1-score   support

        none       0.70      0.85      0.77        87
    implicit       0.46      0.26      0.33        42

    accuracy                           0.66       129
   macro avg       0.58      0.56      0.55       129
weighted avg       0.62      0.66      0.63       129



In [23]:
uncertainty_eval_df = task_a_df[
    task_a_df["predicted_uncertainty_level"].isin(["low", "medium", "high"])
].copy()

print("Uncertainty eval examples:", len(uncertainty_eval_df))

y_true_unc = uncertainty_eval_df["gold_uncertainty_level"].tolist()
y_pred_unc = uncertainty_eval_df["predicted_uncertainty_level"].tolist()

print("Uncertainty accuracy:", accuracy_score(y_true_unc, y_pred_unc))

print("\nUncertainty classification report:")
print(
    classification_report(
        y_true_unc,
        y_pred_unc,
        labels=["low", "medium", "high"],
        zero_division=0,
    )
)

Uncertainty eval examples: 134
Uncertainty accuracy: 0.8507462686567164

Uncertainty classification report:
              precision    recall  f1-score   support

         low       0.86      0.99      0.92       115
      medium       0.00      0.00      0.00        14
        high       0.00      0.00      0.00         5

    accuracy                           0.85       134
   macro avg       0.29      0.33      0.31       134
weighted avg       0.74      0.85      0.79       134



#### Task B 

In [24]:
#TASK B EVALUATION

def lcs_length(x: List[str], y: List[str]) -> int:
    m, n = len(x), len(y)

    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m):
        for j in range(n):
            if x[i] == y[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[m][n]


def rouge_l_score(prediction: str, reference: str) -> float:
    pred_tokens = prediction.lower().split()
    ref_tokens = reference.lower().split()

    if not pred_tokens or not ref_tokens:
        return 0.0

    lcs = lcs_length(pred_tokens, ref_tokens)

    precision = lcs / len(pred_tokens)
    recall = lcs / len(ref_tokens)

    if precision + recall == 0:
        return 0.0

    f1 = 2 * precision * recall / (precision + recall)

    return f1

In [25]:
generation_eval_records = []

for record in test_records:
    gold_label = record["majority_label"]

    if gold_label in ["premise", "claim"]:
        reference = record.get("canonical_implicit_text", "")

        if reference:
            generation_eval_records.append(record)

print("Generation eval records:", len(generation_eval_records))

Generation eval records: 42


In [26]:
#inference for task B

task_b_predictions = []

for record in tqdm(generation_eval_records, desc="Task B inference"):
    tweet = record["tweet_text"]
    gold_type = record["majority_label"]
    reference = record["canonical_implicit_text"]

    messages = build_task_b_prompt(
        tweet=tweet,
        implicit_type=gold_type,
        uncertainty_level=record["uncertainty_level"],
        vote_distribution=record["vote_distribution"],
    )

    raw_output = generate_from_messages(
        messages,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=False,
    )

    parsed = try_parse_json(raw_output)

    if parsed is None:
        predicted_argument = ""
        predicted_type = "parse_error"
    else:
        predicted_argument = parsed.get("implicit_argument", "")
        predicted_type = normalize_predicted_label(parsed.get("implicit_type", gold_type))

    rouge_l = rouge_l_score(predicted_argument, reference)

    task_b_predictions.append(
        {
            "id": record["id"],
            "tweet_text": tweet,

            "gold_type": gold_type,
            "gold_reference": reference,
            "gold_all_majority_references": record.get("majority_implicit_texts", []),
            "gold_all_valid_references": record.get("valid_implicit_texts", []),

            "predicted_type": predicted_type,
            "predicted_argument": predicted_argument,

            "rouge_l_canonical": rouge_l,

            "uncertainty_level": record["uncertainty_level"],
            "vote_distribution": record["vote_distribution"],

            "raw_output": raw_output,
            "parse_success": parsed is not None,
        }
    )

Task B inference:   0%|          | 0/42 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\jeann\Documents\Projets_Python\Deep_Learning\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
c:\Users\jeann\Documents\Projets_Python\Deep_Learning\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
 

In [27]:
task_b_df = pd.DataFrame(task_b_predictions)

display(task_b_df.head())

print("Parse success:")
display(task_b_df["parse_success"].value_counts())

print("Mean ROUGE-L:")
print(task_b_df["rouge_l_canonical"].mean())

,id,tweet_text,gold_type,gold_reference,gold_all_majority_references,gold_all_valid_references,predicted_type,predicted_argument,rouge_l_canonical,uncertainty_level,vote_distribution,raw_output,parse_success
0,1145,It isn't funny. Democrats have been forcefully...,premise,If {Democrats have been forcefully controlling...,[If {Democrats have been forcefully controllin...,[If {Democrats have been forcefully controllin...,premise,Democrat's policies are forced and they contro...,0.062500,low,"{'none': 0.0, 'premise': 1.0, 'claim': 0.0}","{\n ""implicit_type"": ""premise"",\n ""implicit_...",True
1,3013,It's revealing that lying about his asylum cla...,claim,Emad Al Swealmeen should be removed,"[Emad Al Swealmeen should be removed, Emad Al ...","[Emad Al Swealmeen should be removed, Emad Al ...",claim,making false claims should be sufficient to re...,0.266667,low,"{'none': 0.0, 'premise': 0.0, 'claim': 1.0}","{\n ""implicit_type"": ""claim"",\n ""implicit_ar...",True
2,1942,And let's not forget the GQP are the same peop...,premise,Crying about masks and then mandating vaccines...,[Crying about masks and then mandating vaccine...,[Crying about masks and then mandating vaccine...,premise,If you support mask mandates but then fear tha...,0.068966,low,"{'none': 0.0, 'premise': 0.8, 'claim': 0.2}","{\n ""implicit_type"": ""premise"",\n ""implicit_...",True
3,1623,When anybody mentions the word vaccine I run t...,premise,"when you steal a concept to misuse it, you des...","[when you steal a concept to misuse it, you de...","[when you steal a concept to misuse it, you de...",premise,It is good to run away from anything that has ...,0.117647,low,"{'none': 0.2, 'premise': 0.8, 'claim': 0.0}","{\n ""implicit_type"": ""premise"",\n ""implicit_...",True
4,398,You want to force experimental vaccines on pre...,premise,Forcing experimental vaccines on pregnant wome...,[Forcing experimental vaccines on pregnant wom...,[Forcing experimental vaccines on pregnant wom...,premise,Forcing experimental vaccines on pregnant wome...,0.875000,low,"{'none': 0.2, 'premise': 0.8, 'claim': 0.0}","{\n ""implicit_type"": ""premise"",\n ""implicit_...",True


Parse success:


parse_success
True    42
Name: count, dtype: int64

Mean ROUGE-L:
0.22575437045685146


In [28]:
#ROUGE-L against multiple references

def max_rouge_l_against_references(prediction: str, references: List[str]) -> float:
    references = [ref for ref in references if isinstance(ref, str) and ref.strip()]

    if not references:
        return 0.0

    return max(
        rouge_l_score(prediction, ref)
        for ref in references
    )

In [29]:
task_b_df["rouge_l_max_valid_refs"] = task_b_df.apply(
    lambda row: max_rouge_l_against_references(
        row["predicted_argument"],
        row["gold_all_valid_references"],
    ),
    axis=1,
)

task_b_df["rouge_l_max_majority_refs"] = task_b_df.apply(
    lambda row: max_rouge_l_against_references(
        row["predicted_argument"],
        row["gold_all_majority_references"],
    ),
    axis=1,
)

print("Mean ROUGE-L canonical:")
print(task_b_df["rouge_l_canonical"].mean())

print("\nMean ROUGE-L max majority refs:")
print(task_b_df["rouge_l_max_majority_refs"].mean())

print("\nMean ROUGE-L max valid refs:")
print(task_b_df["rouge_l_max_valid_refs"].mean())

Mean ROUGE-L canonical:
0.22575437045685146

Mean ROUGE-L max majority refs:
0.30699549821573224

Mean ROUGE-L max valid refs:
0.3101220013422354


In [30]:
#Save task B predictions 

task_b_predictions_path = EVAL_DIR / "task_b_predictions.csv"
task_b_df.to_csv(task_b_predictions_path, index=False)

print("Saved:", task_b_predictions_path)

Saved: evaluation_results\task_b_predictions.csv


#### Task C : full pipeline

In [31]:
#Full pipeline evaluation

def predict_full_pipeline(tweet: str) -> Dict[str, Any]:
    task_a_messages = build_task_a_prompt(tweet)

    task_a_raw = generate_from_messages(
        task_a_messages,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False,
    )

    task_a_json = try_parse_json(task_a_raw)

    if task_a_json is None:
        return {
            "parse_success_task_a": False,
            "predicted_implicit_type": "parse_error",
            "predicted_implicit_status": "parse_error",
            "predicted_uncertainty_level": "unknown",
            "predicted_argument": "",
            "task_a_raw": task_a_raw,
            "task_b_raw": "",
        }

    predicted_type = normalize_predicted_label(
        task_a_json.get("implicit_type", "none")
    )

    predicted_uncertainty = task_a_json.get(
        "uncertainty_level",
        "unknown",
    )

    predicted_vote_distribution = task_a_json.get(
        "vote_distribution",
        {
            "none": 0.0,
            "premise": 0.0,
            "claim": 0.0,
        },
    )

    if predicted_type in ["none", "ambiguous"]:
        return {
            "parse_success_task_a": True,
            "parse_success_task_b": None,
            "predicted_implicit_type": predicted_type,
            "predicted_implicit_status": task_a_json.get("implicit_status", ""),
            "predicted_uncertainty_level": predicted_uncertainty,
            "predicted_vote_distribution": predicted_vote_distribution,
            "predicted_argument": "",
            "task_a_raw": task_a_raw,
            "task_b_raw": "",
        }

    task_b_messages = build_task_b_prompt(
        tweet=tweet,
        implicit_type=predicted_type,
        uncertainty_level=predicted_uncertainty,
        vote_distribution=predicted_vote_distribution,
    )

    task_b_raw = generate_from_messages(
        task_b_messages,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=False,
    )

    task_b_json = try_parse_json(task_b_raw)

    if task_b_json is None:
        predicted_argument = ""
        parse_success_task_b = False
    else:
        predicted_argument = task_b_json.get("implicit_argument", "")
        parse_success_task_b = True

    return {
        "parse_success_task_a": True,
        "parse_success_task_b": parse_success_task_b,
        "predicted_implicit_type": predicted_type,
        "predicted_implicit_status": task_a_json.get("implicit_status", ""),
        "predicted_uncertainty_level": predicted_uncertainty,
        "predicted_vote_distribution": predicted_vote_distribution,
        "predicted_argument": predicted_argument,
        "task_a_raw": task_a_raw,
        "task_b_raw": task_b_raw,
    }

In [32]:
#full pipeline inference 


full_pipeline_predictions = []

for record in tqdm(test_records, desc="Full pipeline inference"):
    tweet = record["tweet_text"]

    prediction = predict_full_pipeline(tweet)

    gold_reference = record.get("canonical_implicit_text", "")

    if gold_reference and prediction.get("predicted_argument", ""):
        rouge_l = rouge_l_score(
            prediction["predicted_argument"],
            gold_reference,
        )
    else:
        rouge_l = 0.0

    full_pipeline_predictions.append(
        {
            "id": record["id"],
            "tweet_text": tweet,

            "gold_majority_label": record["majority_label"],
            "gold_implicit_status": record["implicit_status"],
            "gold_uncertainty_level": record["uncertainty_level"],
            "gold_reference": gold_reference,
            "gold_all_valid_references": record.get("valid_implicit_texts", []),
            "usable_for_classification": record.get("usable_for_classification", False),

            **prediction,

            "rouge_l_canonical": rouge_l,
        }
    )

Full pipeline inference:   0%|          | 0/134 [00:00<?, ?it/s]

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\jeann\Documents\Projets_Python\Deep_Learning\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
c:\Users\jeann\Documents\Projets_Python\Deep_Learning\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
 

In [33]:

full_pipeline_df = pd.DataFrame(full_pipeline_predictions)

display(full_pipeline_df.head())

,id,tweet_text,gold_majority_label,gold_implicit_status,gold_uncertainty_level,gold_reference,gold_all_valid_references,usable_for_classification,parse_success_task_a,parse_success_task_b,predicted_implicit_type,predicted_implicit_status,predicted_uncertainty_level,predicted_vote_distribution,predicted_argument,task_a_raw,task_b_raw,rouge_l_canonical
0,1145,It isn't funny. Democrats have been forcefully...,premise,yes,low,If {Democrats have been forcefully controlling...,[If {Democrats have been forcefully controllin...,True,True,None,none,no,low,"{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",,"{\n ""argument_analysis"": ""The annotators most...",,0.0
1,3298,I sure hope everyone is paying attention.There...,none,no,low,,[there is a problem with illegal immigrants ha...,True,True,None,none,no,low,"{'none': 0.8, 'premise': 0.2, 'claim': 0.0}",,"{\n ""argument_analysis"": ""The annotators most...",,0.0
2,3214,Refugees resettled in the UK under the Vulnera...,none,no,low,,[],True,True,None,none,no,low,"{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",,"{\n ""argument_analysis"": ""The annotators most...",,0.0
3,3013,It's revealing that lying about his asylum cla...,claim,yes,low,Emad Al Swealmeen should be removed,"[Emad Al Swealmeen should be removed, Emad Al ...",True,True,None,none,no,low,"{'none': 0.8, 'premise': 0.2, 'claim': 0.0}",,"{\n ""argument_analysis"": ""The annotators most...",,0.0
4,2129,BXP and Tory's should not endorse mass migrati...,none,no,low,,[],True,True,None,none,no,low,"{'none': 1.0, 'premise': 0.0, 'claim': 0.0}",,"{\n ""argument_analysis"": ""The annotators most...",,0.0


In [34]:
#full pipeline classification metrics

full_hard_eval_df = full_pipeline_df[
    full_pipeline_df["usable_for_classification"] == True
].copy()

full_hard_eval_df = full_hard_eval_df[
    full_hard_eval_df["predicted_implicit_type"].isin(["none", "premise", "claim"])
].copy()

y_true_full = full_hard_eval_df["gold_majority_label"].tolist()
y_pred_full = full_hard_eval_df["predicted_implicit_type"].tolist()

labels = ["none", "premise", "claim"]

print("Full pipeline classification accuracy:")
print(accuracy_score(y_true_full, y_pred_full))

print("\nFull pipeline classification report:")
print(
    classification_report(
        y_true_full,
        y_pred_full,
        labels=labels,
        zero_division=0,
    )
)

Full pipeline classification accuracy:
0.6434108527131783

Full pipeline classification report:
              precision    recall  f1-score   support

        none       0.70      0.85      0.77        87
     premise       0.56      0.24      0.34        37
       claim       0.00      0.00      0.00         5

    accuracy                           0.64       129
   macro avg       0.42      0.36      0.37       129
weighted avg       0.64      0.64      0.62       129



In [35]:
#full pipeline generation metrics

full_pipeline_df["rouge_l_max_valid_refs"] = full_pipeline_df.apply(
    lambda row: max_rouge_l_against_references(
        row["predicted_argument"],
        row["gold_all_valid_references"],
    ),
    axis=1,
)

generation_rows = full_pipeline_df[
    full_pipeline_df["gold_majority_label"].isin(["premise", "claim"])
].copy()

print("Full pipeline generation rows:", len(generation_rows))

print("\nMean ROUGE-L canonical:")
print(generation_rows["rouge_l_canonical"].mean())

print("\nMean ROUGE-L max valid refs:")
print(generation_rows["rouge_l_max_valid_refs"].mean())

Full pipeline generation rows: 42

Mean ROUGE-L canonical:
0.049797274001166084

Mean ROUGE-L max valid refs:
0.07147254287778748


In [36]:
#full pipeline predictions 

full_pipeline_predictions_path = EVAL_DIR / "full_pipeline_predictions.csv"
full_pipeline_df.to_csv(full_pipeline_predictions_path, index=False)

print("Saved:", full_pipeline_predictions_path)

Saved: evaluation_results\full_pipeline_predictions.csv


### Save a sample for human evaluation

In [37]:
#SAMPLE for human evaluation

human_eval_sample = full_pipeline_df.sample(
    n=min(30, len(full_pipeline_df)),
    random_state=42,
).copy()

human_eval_columns = [
    "id",
    "tweet_text",
    "gold_majority_label",
    "gold_reference",
    "predicted_implicit_type",
    "predicted_argument",
    "predicted_uncertainty_level",
    "rouge_l_max_valid_refs",
]

human_eval_sample = human_eval_sample[human_eval_columns]

human_eval_path = EVAL_DIR / "human_eval_sample.csv"
human_eval_sample.to_csv(human_eval_path, index=False)

print("Saved:", human_eval_path)

human_eval_sample.head()

Saved: evaluation_results\human_eval_sample.csv


,id,tweet_text,gold_majority_label,gold_reference,predicted_implicit_type,predicted_argument,predicted_uncertainty_level,rouge_l_max_valid_refs
127,325,you were punishing people for not wearing mask...,premise,punishing people for not wearing masks and get...,none,,low,0.0
66,415,"Bodily autonomy, except when it's a brand new ...",none,,none,,low,0.0
104,397,The abortionist have lost the right to kill ba...,none,,none,,low,0.0
19,211,"Reagan was great, but every President makes a ...",none,,none,,low,0.0
42,676,Forced experimental vaccines who doesn't work ...,none,,none,,low,0.0


### Save metrics

In [38]:
#Final metrics summary

metrics_summary = {}

# Task A hard classification
if len(hard_eval_df) > 0:
    metrics_summary["task_a_accuracy"] = accuracy_score(y_true, y_pred)
else:
    metrics_summary["task_a_accuracy"] = None

# Binary classification
if len(binary_eval_df) > 0:
    metrics_summary["task_a_binary_accuracy"] = accuracy_score(y_true_bin, y_pred_bin)
else:
    metrics_summary["task_a_binary_accuracy"] = None

# Uncertainty
if len(uncertainty_eval_df) > 0:
    metrics_summary["uncertainty_accuracy"] = accuracy_score(y_true_unc, y_pred_unc)
else:
    metrics_summary["uncertainty_accuracy"] = None

# Task B generation
if len(task_b_df) > 0:
    metrics_summary["task_b_mean_rouge_l_canonical"] = float(task_b_df["rouge_l_canonical"].mean())
    metrics_summary["task_b_mean_rouge_l_max_majority_refs"] = float(task_b_df["rouge_l_max_majority_refs"].mean())
    metrics_summary["task_b_mean_rouge_l_max_valid_refs"] = float(task_b_df["rouge_l_max_valid_refs"].mean())
else:
    metrics_summary["task_b_mean_rouge_l_canonical"] = None
    metrics_summary["task_b_mean_rouge_l_max_majority_refs"] = None
    metrics_summary["task_b_mean_rouge_l_max_valid_refs"] = None

# Full pipeline
if len(full_hard_eval_df) > 0:
    metrics_summary["full_pipeline_classification_accuracy"] = accuracy_score(y_true_full, y_pred_full)
else:
    metrics_summary["full_pipeline_classification_accuracy"] = None

if len(generation_rows) > 0:
    metrics_summary["full_pipeline_mean_rouge_l_canonical"] = float(generation_rows["rouge_l_canonical"].mean())
    metrics_summary["full_pipeline_mean_rouge_l_max_valid_refs"] = float(generation_rows["rouge_l_max_valid_refs"].mean())
else:
    metrics_summary["full_pipeline_mean_rouge_l_canonical"] = None
    metrics_summary["full_pipeline_mean_rouge_l_max_valid_refs"] = None

metrics_summary

{'task_a_accuracy': 0.6434108527131783,
 'task_a_binary_accuracy': 0.6589147286821705,
 'uncertainty_accuracy': 0.8507462686567164,
 'task_b_mean_rouge_l_canonical': 0.22575437045685146,
 'task_b_mean_rouge_l_max_majority_refs': 0.30699549821573224,
 'task_b_mean_rouge_l_max_valid_refs': 0.3101220013422354,
 'full_pipeline_classification_accuracy': 0.6434108527131783,
 'full_pipeline_mean_rouge_l_canonical': 0.049797274001166084,
 'full_pipeline_mean_rouge_l_max_valid_refs': 0.07147254287778748}

In [39]:
metrics_path = EVAL_DIR / "metrics_summary.json"

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2)

print("Saved:", metrics_path)

Saved: evaluation_results\metrics_summary.json


In [40]:
#check errors

classification_errors = full_hard_eval_df[
    full_hard_eval_df["gold_majority_label"] != full_hard_eval_df["predicted_implicit_type"]
].copy()

print("Classification errors:", len(classification_errors))

classification_errors[
    [
        "id",
        "tweet_text",
        "gold_majority_label",
        "predicted_implicit_type",
        "gold_uncertainty_level",
        "predicted_uncertainty_level",
        "task_a_raw",
    ]
].head(10)

Classification errors: 46


,id,tweet_text,gold_majority_label,predicted_implicit_type,gold_uncertainty_level,predicted_uncertainty_level,task_a_raw
0,1145,It isn't funny. Democrats have been forcefully...,premise,none,low,low,"{\n ""argument_analysis"": ""The annotators most..."
3,3013,It's revealing that lying about his asylum cla...,claim,none,low,low,"{\n ""argument_analysis"": ""The annotators most..."
6,26,Happy World Refrigeration Day to everyone work...,none,claim,low,low,"{\n ""argument_analysis"": ""The tweet provides ..."
7,2169,"So is the UK going to stand up for an actual, ...",none,premise,low,low,"{\n ""argument_analysis"": ""The tweet contains ..."
9,2113,We need a Migration Control Department to over...,none,claim,low,medium,"{\n ""argument_analysis"": ""The tweet provides ..."
10,1942,And let's not forget the GQP are the same peop...,premise,none,low,low,"{\n ""argument_analysis"": ""The annotators most..."
15,398,You want to force experimental vaccines on pre...,premise,none,low,low,"{\n ""argument_analysis"": ""The annotators most..."
17,253,With me discuss the many problems of the covid...,premise,none,medium,low,"{\n ""argument_analysis"": ""The annotators most..."
20,2016,Is this a problem with the mRNA vaccine? Natur...,premise,none,low,low,"{\n ""argument_analysis"": ""The annotators most..."
27,1831,This is guaranteed? Sophie Trudeau and million...,premise,claim,low,low,"{\n ""argument_analysis"": ""The tweet provides ..."


In [41]:
classification_errors = full_hard_eval_df[
    full_hard_eval_df["gold_majority_label"] != full_hard_eval_df["predicted_implicit_type"]
].copy()

print("Classification errors:", len(classification_errors))

classification_errors[
    [
        "id",
        "tweet_text",
        "gold_majority_label",
        "predicted_implicit_type",
        "gold_uncertainty_level",
        "predicted_uncertainty_level",
        "task_a_raw",
    ]
].head(10)

Classification errors: 46


,id,tweet_text,gold_majority_label,predicted_implicit_type,gold_uncertainty_level,predicted_uncertainty_level,task_a_raw
0,1145,It isn't funny. Democrats have been forcefully...,premise,none,low,low,"{\n ""argument_analysis"": ""The annotators most..."
3,3013,It's revealing that lying about his asylum cla...,claim,none,low,low,"{\n ""argument_analysis"": ""The annotators most..."
6,26,Happy World Refrigeration Day to everyone work...,none,claim,low,low,"{\n ""argument_analysis"": ""The tweet provides ..."
7,2169,"So is the UK going to stand up for an actual, ...",none,premise,low,low,"{\n ""argument_analysis"": ""The tweet contains ..."
9,2113,We need a Migration Control Department to over...,none,claim,low,medium,"{\n ""argument_analysis"": ""The tweet provides ..."
10,1942,And let's not forget the GQP are the same peop...,premise,none,low,low,"{\n ""argument_analysis"": ""The annotators most..."
15,398,You want to force experimental vaccines on pre...,premise,none,low,low,"{\n ""argument_analysis"": ""The annotators most..."
17,253,With me discuss the many problems of the covid...,premise,none,medium,low,"{\n ""argument_analysis"": ""The annotators most..."
20,2016,Is this a problem with the mRNA vaccine? Natur...,premise,none,low,low,"{\n ""argument_analysis"": ""The annotators most..."
27,1831,This is guaranteed? Sophie Trudeau and million...,premise,claim,low,low,"{\n ""argument_analysis"": ""The tweet provides ..."


In [43]:
task_b_df_sorted = task_b_df.sort_values(
    by="rouge_l_max_valid_refs",
    ascending=False,
)
print("Best generation examples:")
display(
    task_b_df_sorted[
        [
            "id",
            "tweet_text",
            "gold_type",
            "gold_reference",
            "predicted_argument",
            "rouge_l_max_valid_refs",
        ]
    ].tail(10)
)

Best generation examples:


,id,tweet_text,gold_type,gold_reference,predicted_argument,rouge_l_max_valid_refs
37,2620,Turkish police force Syrian refugees to #Greek...,premise,If Turkish police force Syrian refugees to #Gr...,If you force people to come to your country th...,0.187500
14,680,I remember when the 'vaccine' was first rolled...,premise,"If X and Y happen at the same time, X causes Y",A vaccine cannot be trusted if it causes probl...,0.166667
2,1942,And let's not forget the GQP are the same peop...,premise,Crying about masks and then mandating vaccines...,If you support mask mandates but then fear tha...,0.153846
16,1703,Life isn't a movie and vaccines aren't created...,premise,If Life isn't a movie and vaccines aren't crea...,A vaccine must be effective to be good,0.142857
10,3283,What's your problem with people immigrating to...,premise,People whose ancestors engaged in mass immigra...,immigrants are not like other immigrants,0.125000
3,1623,When anybody mentions the word vaccine I run t...,premise,"when you steal a concept to misuse it, you des...",It is good to run away from anything that has ...,0.117647
33,695,the government never said YOU HAVE to get the ...,premise,a comparison between two situations is invalid...,It is wrong to compare mandating vaccines to m...,0.117647
0,1145,It isn't funny. Democrats have been forcefully...,premise,If {Democrats have been forcefully controlling...,Democrat's policies are forced and they contro...,0.114286
15,1232,I think it was hope that the vaccine would wor...,premise,Public health officials who impose mandates ba...,mandating vaccine because of hope is wrong,0.111111
19,1053,People selling cocaine would take cocaine. Doc...,premise,if you don't take the vaccine yourself you are...,Doctors and politicians are complicit in selli...,0.100000
